# tinygpt on Colab (L4): data, tokenizer, training, Gradio demo

Data: [hchautran/vietnamese-wiki-segmented](https://huggingface.co/datasets/hchautran/vietnamese-wiki-segmented), 1.27M articles, column `segmented_text`. Words are joined by underscores (`tổ_chức quốc_tế`), punctuation is already stripped, list items appear as the word `BULLET`.

Runtime: **Runtime > Change runtime type > L4 GPU** (Colab Pro). Config: `configs/vi_wiki_colab_l4.json`, ~60M params, context 512, TF32 matmuls.

Tokenizer: 8192-token BPE with the whole-word pattern (`WHOLE_WORD_BPE = True`): `tổ_chức` is one pre-token, so frequent words become single tokens. Set it to `False` for the GPT-2 pattern.

Google Drive keeps everything that must survive a disconnect: `DRIVE_ROOT/tokenizers/` (BPE trained here), `DRIVE_ROOT/cache/` (token `.npy` files), `DRIVE_ROOT/runs/` (checkpoints + CSV). Each checkpoint is ~700MB (weights + AdamW state); a full run writes 7 of them (~5GB).

**Before the first run:** run `uv run pytest` locally, then push this repo.

**Demo only (training already done):** run sections 1 and 2's first cell, then jump to section 4.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/YOUR_USER/tinygpt.git"  # private repo: add a GITHUB_TOKEN secret (key icon, left bar)
DRIVE_ROOT = "/content/drive/MyDrive/tinygpt"          # survives disconnects: tokenizer, token cache, runs
CONFIG = "configs/vi_wiki_colab_l4.json"

DATASET = "hchautran/vietnamese-wiki-segmented"
TEXT_COLUMN = "segmented_text"
# "keep": train on "tổ_chức quốc_tế" (word boundaries visible to the model; the demo segments prompts with underthesea)
# "join": train on "tổ chức quốc tế" (plain Vietnamese without punctuation; prompts used as typed)
SEG_MODE = "keep"
TRAIN_MB = 2000      # more than the dataset holds (~1.3GB): take all of it
MIN_CHARS = 200      # drops stub and template-only pages
WHOLE_WORD_BPE = True  # "tổ_chức" as one pre-token (WORD_PRETOKEN_PATTERN); False = GPT-2 pattern
BPE_VOCAB = 8192     # room for whole words as single tokens; embedding + head ≈ 10.5M params at d_model 640
BPE_SAMPLE_MB = 100  # BPE is trained on the first 100MB of the train file

DATA_NAME = "vi_wiki_seg" if SEG_MODE == "keep" else "vi_wiki_seg_join"
# The tokenizer name keys the token cache: a different pattern must never reuse another's cache.
TOKENIZER = f"{DATA_NAME}_{'words_' if WHOLE_WORD_BPE else ''}bpe{BPE_VOCAB}"
RUN_NAME = f"{TOKENIZER}_l4"

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
for sub in ("tokenizers", "cache", "runs"):
    Path(DRIVE_ROOT, sub).mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(DRIVE_ROOT, "runs", RUN_NAME)

REPO_DIR = Path("/content/tinygpt")
if not REPO_DIR.exists():
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass  # public repo, or no secret set
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    r = subprocess.run(["git", "clone", "--depth", "1", url, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode != 0:  # never echo the URL: it may hold the token
        raise RuntimeError("git clone failed:\n" + (r.stderr.replace(token, "***") if token else r.stderr))
os.chdir(REPO_DIR)
!git log --oneline -1

In [ ]:
# --no-deps keeps Colab's preinstalled CUDA torch instead of pulling the pinned one from pyproject.toml.
!pip install -q -e . --no-deps
!pip install -q regex einops einx jaxtyping pyarrow gradio huggingface_hub underthesea

import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print("torch", torch.__version__, "|", gpu)
if "L4" not in gpu:
    print(f"!! expected an L4, got {gpu}: the config's model size and step count assume L4 speed")

# TF32: fp32 matmuls run on tensor cores (Ampere/Ada GPUs like L4; no effect on T4).
# In this process (speed test, demo) and, via the env var, in the `!python scripts/train.py` subprocess.
torch.set_float32_matmul_precision("high")
os.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"

## 2. Data and tokenizer

Pipeline: parquet (Hugging Face) → `prepare_vi.py --segmented` → `data/<DATA_NAME>/vi_{train,valid}.txt` → `train_bpe.py` on a 100MB sample → `tokenize_data.py` → `data/cache/*.npy` (uint16, memmapped by training).

The tokenizer and token cache are backed up to Drive and restored on later sessions; download, text preparation and BPE training are then skipped. Each cell below skips itself when its output already exists.

In [ ]:
import json
import shutil
import sys

sys.path.insert(0, "scripts")
import train as train_script  # scripts/train.py: load_config drops "_" keys, makes betas a tuple

# Base config + this notebook's data choice, written to one file that train.py reads.
cfg = train_script.load_config(Path(CONFIG))
cfg["train_path"] = f"data/{DATA_NAME}/vi_train.txt"
cfg["val_path"] = f"data/{DATA_NAME}/vi_valid.txt"
cfg["tokenizer_spec"] = f"data/tokenizers/{TOKENIZER}"
RUN_CONFIG = Path("/content/run_config.json")
RUN_CONFIG.write_text(json.dumps(cfg, indent=2))

train_txt, val_txt = Path(cfg["train_path"]), Path(cfg["val_path"])
tok_local = Path(cfg["tokenizer_spec"])
tok_drive = Path(DRIVE_ROOT, "tokenizers", TOKENIZER)
cache_local = Path(cfg["cache_dir"])
cache_drive = Path(DRIVE_ROOT, "cache")
cache_local.mkdir(parents=True, exist_ok=True)

if not tok_local.exists() and tok_drive.exists():
    print("restoring tokenizer", TOKENIZER)
    shutil.copytree(tok_drive, tok_local)

# Batches read random offsets of a memmap. Over Drive's network filesystem that is slow,
# so training always reads a copy on the VM's local disk.
for f in cache_drive.glob(f"{DATA_NAME}_*_{TOKENIZER}_*.npy"):
    if not (cache_local / f.name).exists():
        print("restoring", f.name)
        shutil.copy(f, cache_local / f.name)

def is_cached(txt: Path) -> bool:  # same naming as tinygpt.data.load_tokens
    return any(cache_local.glob(f"{txt.parent.name}_{txt.stem}_{TOKENIZER}_*.npy"))

have_tok = tok_local.exists()
cached = is_cached(train_txt) and is_cached(val_txt)
print(f"tokenizer: {'found' if have_tok else 'missing'} | token cache: {'found' if cached else 'missing'}")

In [ ]:
# Text files: needed only to train the tokenizer or build the token cache.
if not (have_tok and cached) and not train_txt.exists():
    import pyarrow.parquet as pq
    from huggingface_hub import snapshot_download

    raw = Path(snapshot_download(DATASET, repo_type="dataset", local_dir="data/vi_wiki_seg_raw",
                                 allow_patterns=["*.parquet"]))
    parquets = sorted(raw.rglob("*.parquet"))
    columns = pq.ParquetFile(parquets[0]).schema_arrow.names
    print(len(parquets), "parquet files, columns:", columns)
    assert TEXT_COLUMN in columns, "set TEXT_COLUMN (settings cell) to one of the columns above"

    files = " ".join(str(p) for p in parquets)
    !python scripts/prepare_vi.py {files} --text-column {TEXT_COLUMN} --segmented {SEG_MODE} --out-dir {train_txt.parent} --name vi --train-mb {TRAIN_MB} --min-chars {MIN_CHARS}

In [ ]:
# BPE trained on this corpus: the plain-wiki tokenizer never saw "_" and spent merges on punctuation.
if not have_tok:
    EOT = b"<|endoftext|>"
    with open(train_txt, "rb") as f:
        head = f.read(BPE_SAMPLE_MB * 2**20)
    sample = train_txt.with_name("bpe_sample.txt")
    sample.write_bytes(head[: head.rfind(EOT) + len(EOT)])  # whole documents only

    pattern = "words" if WHOLE_WORD_BPE else "gpt2"
    !python scripts/train_bpe.py {sample} --vocab-size {BPE_VOCAB} --name {TOKENIZER} --pattern {pattern}
    assert (tok_local / "meta.json").exists(), "train_bpe.py failed: see its message above"
    shutil.copytree(tok_local, tok_drive, dirs_exist_ok=True)
    have_tok = True
print(json.loads((tok_local / "meta.json").read_text()))

In [ ]:
if not cached:
    !python scripts/tokenize_data.py {RUN_CONFIG}
    for f in cache_local.glob(f"{DATA_NAME}_*_{TOKENIZER}_*.npy"):
        if not (cache_drive / f.name).exists():
            print("backing up", f.name, "to Drive")
            shutil.copy(f, cache_drive / f.name)
    cached = is_cached(train_txt) and is_cached(val_txt)
assert cached

from tinygpt.tokenizer import load_tokenizer
tok = load_tokenizer(cfg["tokenizer_spec"])
sample_ids = tok.encode("Hà_Nội là thủ_đô của Việt_Nam" if SEG_MODE == "keep" else "Hà Nội là thủ đô của Việt Nam")
print(len(sample_ids), "tokens:", [tok.decode([i]) for i in sample_ids])

## 3. Training

First a 200-step speed test (same config, nothing saved to Drive) to measure s/step and peak memory. Out of memory → lower `batch_size` in the config and rerun. Then set `max_steps` so the full run fits your session: the default (36k steps, ~590M tokens) is sized for roughly 4–6h on an L4 with TF32, an estimate to replace with the measured number.

`train()` cannot resume yet: a disconnect means restarting from step 0. Colab Pro sessions last longer than free ones, but keep the tab open.

In [ ]:
from tinygpt.train import train

SPEED_STEPS = 200
torch.cuda.reset_peak_memory_stats()
out = train(**{**cfg, "max_steps": SPEED_STEPS, "warmup_iters": 20, "log_every": 50,
               "eval_every": 10**9, "ckpt_every": 10**9}, run_dir="/content/speedtest")
s_per_step = out["elapsed_s"] / SPEED_STEPS
print(f"{s_per_step:.3f} s/step | peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
print(f"full run: {cfg['max_steps']:,} steps ≈ {cfg['max_steps'] * s_per_step / 3600:.1f} h, "
      f"{cfg['max_steps'] * cfg['batch_size'] * cfg['context_length'] / 1e6:.0f}M tokens")
torch.cuda.empty_cache()  # hand the memory back before the real run's process starts

In [ ]:
OVERWRITE = False  # no resume yet: a rerun starts at step 0 and overwrites train_log.csv
if any(RUN_DIR.glob("ckpt_*.pt")) and not OVERWRITE:
    raise RuntimeError(f"{RUN_DIR} already has checkpoints: set OVERWRITE = True or change RUN_NAME")

!python scripts/train.py {RUN_CONFIG} --run-dir {RUN_DIR}

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

log = pd.read_csv(RUN_DIR / "train_log.csv")
val = log.dropna(subset=["val_loss"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(log.step, log.train_loss, alpha=0.6, label="train")
ax1.plot(val.step, val.val_loss, "o-", label="val")
ax1.set(xlabel="step", ylabel="loss", title="Loss")
ax1.legend()
ax2.plot(log.step, log.lr)
ax2.set(xlabel="step", ylabel="learning rate", title="Warmup + cosine schedule")
plt.show()
print(f"final val loss {val.val_loss.iloc[-1]:.3f} at step {val.step.iloc[-1]}")

## 4. Gradio demo

Uses the same `load_model` as `scripts/generate.py` and your `tinygpt.generate.generate`. The checkpoint dropdown shows how samples improve over training.

Prompts are converted to look like the training text: punctuation removed, and with `SEG_MODE = "keep"`, words segmented by underthesea (`Việt Nam là` → `Việt_Nam là`). Output underscores are shown as spaces.

`share=True` prints a public `*.gradio.live` link: anyone with it can use the demo while this notebook runs.

In [ ]:
import codecs
import re
import sys
import unicodedata

import gradio as gr
import torch

sys.path.insert(0, "scripts")
import generate as generate_cli  # scripts/generate.py: load_model, pick_device
from prepare_vi import STRAY_UNDERSCORE  # same "_ only between letters" rule as the training text
from tinygpt.generate import generate
from tinygpt.tokenizer import load_tokenizer

if SEG_MODE == "keep":
    from underthesea import word_tokenize

PUNCT = re.compile(r"[^\w\s]")  # \w keeps Vietnamese letters, digits and "_"

def to_model_text(prompt: str) -> str:
    # NFC first: some keyboards send decomposed accents, which \p{L} in the BPE pattern would split on.
    text = unicodedata.normalize("NFC", prompt)
    text = PUNCT.sub(" ", text)  # the training text has no punctuation
    if SEG_MODE == "keep" and text.strip():
        text = word_tokenize(text, format="text")  # "Việt Nam là" -> "Việt_Nam là"
        text = STRAY_UNDERSCORE.sub(" ", text)     # never join digits, no edge underscores
    else:
        text = text.replace("_", " ")
    return " ".join(text.split())

def to_display(text: str) -> str:
    return text.replace("_", " ") if SEG_MODE == "keep" else text

DEVICE = generate_cli.pick_device()
ckpts = sorted(RUN_DIR.glob("ckpt_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
assert ckpts, f"no checkpoints in {RUN_DIR}"
choices = [p.name for p in ckpts]
_loaded = {}

def get_model(name):
    if name not in _loaded:
        _loaded.clear()  # one model in memory at a time
        model, config, spec = generate_cli.load_model(str(RUN_DIR / name), DEVICE)
        _loaded[name] = (model, config, load_tokenizer(spec))
    return _loaded[name]

def complete(prompt, ckpt_name, temperature, top_p, max_new_tokens, seed):
    model, config, tok = get_model(ckpt_name)
    generator = torch.Generator().manual_seed(int(seed)) if seed is not None and seed >= 0 else None
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")  # a token can end mid-character
    text = to_model_text(prompt)
    for token_id in generate(model, tok.encode(text), max_new_tokens=int(max_new_tokens),
                             temperature=float(temperature), top_p=float(top_p), eot_id=tok.eot_id,
                             context_length=config["context_length"], generator=generator):
        text += decoder.decode(tok.token_bytes(token_id))
        yield to_display(text)
    yield to_display(text + decoder.decode(b"", final=True))

with gr.Blocks(title="tinygpt") as demo:
    gr.Markdown("# tinygpt: Vietnamese Wikipedia\nA small GPT written from scratch (CS336 assignment 1 architecture).")
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label="Prompt", lines=3)
            ckpt = gr.Dropdown(choices, value=choices[-1], label="Checkpoint")
            temperature = gr.Slider(0.0, 1.5, value=0.8, step=0.05, label="Temperature (0 = greedy)")
            top_p = gr.Slider(0.05, 1.0, value=0.9, step=0.05, label="Top-p")
            max_new = gr.Slider(16, 512, value=300, step=16, label="Max new tokens")
            seed = gr.Number(value=0, precision=0, label="Seed (-1 = random)")
            with gr.Row():
                run = gr.Button("Generate", variant="primary")
                stop = gr.Button("Stop")
        output = gr.Textbox(label="Output (punctuation-free, like the training text)", lines=16)
    gr.Examples([["Việt Nam là"], ["Hà Nội là thủ đô của"], ["Trong lịch sử"]], inputs=prompt)
    event = run.click(complete, [prompt, ckpt, temperature, top_p, max_new, seed], output)
    stop.click(None, cancels=[event])

demo.queue().launch(share=True)